In [1]:
!pip install datasets peft accelerate trl adapter-transformers


In [3]:

!pip uninstall -y transformers




Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 40.5 MB/s eta 0:00:00
  Created wheel for adapter-transformers: filename=adapter_transformers-4.0.0-py3-none-any.whl size=2630 sha256=bb2486331dbeffedc208b823dceff9c93a70ee7f0ed78746dabaf33b1465cc8c
  Stored in directory: /root/.cache/pip/wheels/50/98/71/12adabf46fb654d220b0e11b68d827fb5d8d9b960f9cf2b432
Successfully built adapter-transformers
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are i

In [6]:
import os
from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

os.makedirs("data", exist_ok=True)

dataset.to_json("data/dolly_train.json", orient="records", lines=True)

print("Data saved to data/dolly_train.json")

Creating json from Arrow format:   0%|          | 0/16 [00:00<?, ?ba/s]

Data saved to data/dolly_train.json


In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
#from trl import SFTTrainer
from datasets import load_dataset
from transformers import TrainingArguments

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer




In [10]:
from adapters import AutoAdapterModel
model = AutoAdapterModel.from_pretrained("gpt2")
model.add_adapter("sft", config="pfeiffer")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

### Activate adapter training (mandatory)

1. Freezes all GPT-2 weights

2. Unfreezes only adapter MLP weights

3. Inserts adapter into forward pass

In [11]:

model.train_adapter("sft")
model.set_active_adapters("sft")
model = model.cuda()


In [12]:
print(model.adapter_summary())


Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
sft                      bottleneck          894,528       0.719       1       1
--------------------------------------------------------------------------------
Full model                               124,439,808     100.000               0


In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [4]:
import torch

In [15]:
def format_and_tokenize(example):
    text = f"""### Instruction:
{example['instruction']}

### Context:
{example.get('context','')}

### Response:
{example['response']}"""

    enc = tokenizer(
        text,
        truncation=True,
        max_length=128,      # keep memory small
        padding="max_length" # pad to fixed length
    )

    # Important: store labels as a **list of ints**, not a tensor
    enc["labels"] = enc["input_ids"].copy()
    return enc


In [16]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="data/dolly_train.json")["train"]
dataset = dataset.shuffle(seed=42).select(range(1000))  # reduce size

dataset = dataset.map(
    format_and_tokenize,
    remove_columns=dataset.column_names  # remove original columns
)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [17]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)


In [18]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./out",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    fp16=True,
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

trainer.train()


Step,Training Loss
10,4.172300
20,3.887500
30,3.409300
40,3.583700
50,3.234800
60,3.434800
70,3.470100
80,3.275000
90,3.212000
100,3.304200


TrainOutput(global_step=1500, training_loss=2.88023045984904, metrics={'train_runtime': 111.1762, 'train_samples_per_second': 26.984, 'train_steps_per_second': 13.492, 'total_flos': 198030016512000.0, 'train_loss': 2.88023045984904, 'epoch': 3.0})

In [19]:
# 6. Save  only adapter
model.save_adapter("adapter_sft", "sft")

In [21]:
#save complete model
trainer.save_model("full_model_with_adapter")

In [20]:
#load the saved model
model = AutoAdapterModel.from_pretrained("gpt2")
model.load_adapter("adapter_sft", load_as="sft")
model.set_active_adapters("sft")
model.eval() #without this dropput may be active


In [22]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [23]:
#inferencing
inputs = tokenizer("Explain what LoRA fine-tuning is in simple terms for a product manager.", return_tensors="pt").to(model.device)

#outputs = model.generate(**inputs)
outputs = model.generate(
    **inputs,
    max_new_tokens=50,   # 👈 CRITICAL
    do_sample=False
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [25]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Explain what LoRA fine-tuning is in simple terms for a product manager.

### Context

LoRA is a global, open source, open source, and open source software development platform. It is designed to be a platform for developers to build and deploy software in a variety of environments. It is designed to be


In [26]:
#how to prove adapter is getting used? If this prints None → adapter NOT active.
print(model.active_adapters)

Stack[sft]
